In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
#원-핫 변환 함수: 정답 숫자를 [0,0,...,1,...,0] 형태로 바꾸는 도구를 씁니다.

# 1. 데이터 불러오기
#왜 자동으로 나뉘어질까?
#mnist.load_data()는 Keras에 미리 정의된 MNIST 데이터셋 로더라서
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# CNN 입력 형태에 맞게 reshape (28x28 → 28x28x1)
#(높이, 너비, 채널)
#-1 = 전체 개수 자동으로 맞춰라
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# 라벨을 one-hot 인코딩(클래스 10으로 맞추기)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 2. 모델 구성
# 모델을 함수로 만들어 두면, 학습 셀을 다시 실행할 때마다 새 모델로 처음부터 학습함
# (같은 model 객체로 fit을 다시 호출하면 이전 학습 결과를 이어받아 성능이 부풀려짐)
def build_model():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(28,28,1)),
        #ReLU로 음수는 0, 양수만 살립니다.
        #SAME 패딩으로 크기를 유지합니다(28×28 → 28×28).
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        #풀링(축소): 2×2 중 가장 큰 값만 남기며 크기를 절반으로 줄입니다.
        layers.MaxPooling2D((2,2), padding='same'),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2), padding='same'),

        #펴기
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        #패턴 외우기 방지
        layers.Dropout(0.5),
        #결과 내기: 10개의 숫자에 대한 “확률”을 출력합니다. 가장 큰 값이 모델의 선택입니다.
        layers.Dense(10, activation='softmax')
    ])

build_model().summary()

In [ ]:
# 3. 모델 컴파일
#Adam으로 똑똑하게 가중치 조정.
#categorical_crossentropy로 “정답과 예측의 차이”를 계산.
#accuracy(정확도)를 보고 학습이 잘 되는지 확인.

model = build_model()
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4. 학습
#→ 학습용 데이터(60,000장 중 검증용을 뺀 54,000장)를 여러 번 돌려보면서, 예측 확률이 정답에 더 가까워지도록 계속 수정합니다.
#검증은 학습 데이터의 10%(6,000장)를 떼어 사용하고, 테스트 세트는 마지막 평가에만 사용합니다.
model.fit(x_train, y_train,
          batch_size=100,
          epochs=15,
          validation_split=0.1)

# 5. 평가
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("최종 테스트 정확도:", test_acc)